[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IgnatiusEzeani/spatial-humanities-2026/blob/main/workshop/07_compare_and_adjudicate.ipynb)

# 07 · Compare, disagree and adjudicate

**Spatial Humanities 2026 workshop**

Different methods often agree on easy cases and diverge exactly where scholarly judgement matters. This notebook makes disagreement visible and measures the amount of human intervention required.

## Learning goals
- distinguish model agreement from correctness;
- create a consensus annotation without hiding dissent;
- inspect missing-from/voter information;
- record human accept/edit/reject actions append-only;
- separate **review burden** from **correction burden**;
- produce tidy comparison records for later keynote figures.

> **Key message:** There is no single method winner across all dimensions.

In [ ]:
import json, os, pathlib, subprocess, sys, urllib.request

SETUP_URL = (
    "https://raw.githubusercontent.com/IgnatiusEzeani/"
    "spatial-humanities-2026/main/workshop/sh2026_setup.py"
)
local_setup = pathlib.Path("workshop/sh2026_setup.py")
setup_path = local_setup if local_setup.exists() else pathlib.Path("/content/sh2026_setup.py")
if not local_setup.exists():
    urllib.request.urlretrieve(SETUP_URL, setup_path)
if str(setup_path.parent.resolve()) not in sys.path:
    sys.path.insert(0, str(setup_path.parent.resolve()))

import sh2026_setup as sh
ctx = sh.setup()
repo_dir = ctx.project
data_dir = ctx.data
output_dir = ctx.outputs
FAST_MODE = ctx.fast_mode

from workshop_support import display, checkpoint, compare_spans, show_fields, show_journey, show_spans


## 1. A controlled disagreement example

We first use small deterministic records so the mechanics of adjudication are visible and reproducible. These are **teaching records**, not empirical model results.

In [ ]:
from spatio_textual.moe import ModelAnnotation, adjudicate_entities
import pandas as pd

text = "I travelled from Cambridge to London."
cam_start = text.index("Cambridge")
lon_start = text.index("London")

cambridge = {"text": "Cambridge", "label": "GPE", "start_char": cam_start, "end_char": cam_start + len("Cambridge")}
london = {"text": "London", "label": "GPE", "start_char": lon_start, "end_char": lon_start + len("London")}

records = [
    ModelAnnotation("method_A", {"text": text, "entities": [cambridge, london], "telemetry": []}),
    ModelAnnotation("method_B", {"text": text, "entities": [cambridge, london], "telemetry": []}),
    ModelAnnotation("method_C", {"text": text, "entities": [london], "telemetry": []}),
]

adj = adjudicate_entities(records, threshold=2/3)
consensus_rows = [{
    "span": item.get("text"),
    "label": item.get("label"),
    "votes": item.get("vote_count"),
    "vote_ratio": item.get("vote_ratio"),
    "voters": ", ".join(item.get("voters", [])),
    "missing_from": ", ".join(item.get("missing_from", [])),
    "review": item.get("requires_review"),
} for item in adj.consensus["entities"]]
display(pd.DataFrame(consensus_rows))
checkpoint(f"{len(adj.disagreements)} disagreement record(s) remain visible after consensus.")


`Cambridge` can enter the consensus at a two-of-three threshold while still remaining a disagreement because one method missed it. The consensus record therefore retains:

- `vote_count`
- `vote_ratio`
- `voters`
- `missing_from`
- `requires_review`

Consensus should not erase dissent.

## 2. Agreement is not correctness

Three systems can agree and still be wrong because they share training data, gazetteers, assumptions or ontology. Conversely, a minority method may identify something the others cannot represent.

For Spatial Humanities, disagreement is useful diagnostic evidence about:
- span boundaries;
- label inventories;
- ambiguous place resolution;
- implicit relations;
- historical names;
- inferred journey fields.

## 3. Human review as an auditable operation

The SH2026 common schema stores human decisions alongside machine outputs. Accepting, editing and rejecting are different events and should not be collapsed into a single "corrected" flag.

In [ ]:
from spatio_textual.review import apply_human_review, human_correction_burden

machine_journeys = [
    {"journeyId": "j1", "start_location": "Cambridge", "end_location": "London", "human_status": "unreviewed", "human_edits": [], "requires_review": True},
    {"journeyId": "j2", "start_location": "Cambridge", "end_location": "London", "human_status": "unreviewed", "human_edits": [], "requires_review": True},
    {"journeyId": "j3", "start_location": "Cambridge", "end_location": "Paris", "human_status": "unreviewed", "human_edits": [], "requires_review": True},
]

accepted = apply_human_review(machine_journeys[0], action="accept", reason="human_flag")
edited = apply_human_review(machine_journeys[1], action="edit", field="end_location", new_value="London, England", reason="disambiguation")
rejected = apply_human_review(machine_journeys[2], action="reject", reason="unsupported_llm_field")

reviewed = [accepted, edited, rejected]
display(pd.DataFrame([{
    "journey": row["journeyId"],
    "decision": row.get("human_status"),
    "start": row.get("start_location"),
    "end": row.get("end_location"),
    "edits_recorded": len(row.get("human_edits", [])),
    "review_required": row.get("requires_review"),
} for row in reviewed]))


The edit event preserves both the old and new values. This is important for reproducibility: the final clean value alone does not tell us how much human work was needed to obtain it.

In [ ]:
burden = human_correction_burden(reviewed)
show_fields(burden, list(burden))
checkpoint("Review volume and correction volume answer different methodological questions.")


### Review burden vs correction burden

- **Review burden**: how many machine suggestions a human had to inspect.
- **Correction burden**: how many inspected suggestions had to be edited or rejected.

A system that is highly accurate but flags everything may still impose a high review burden. A system with low review volume but frequent edits may impose a high correction burden.

This distinction will be one of the keynote benchmark dimensions.

## 4. Side-by-side method comparison

Not every method should be forced into the same metric. Use `null` when a metric does not apply rather than writing zero.

In [ ]:
comparison = pd.DataFrame([
    ["Manual reference", "Spatial annotation", "Human judgement", "Reference is reviewed, not assumed infallible"],
    ["Rules + gazetteer", "Spatial annotation", "Measured benchmark only", "Deterministic and inspectable"],
    ["Contextual NER", "Toponym recognition", "Measured benchmark only", "Named-entity ontology has a known ceiling"],
    ["LLM journey", "Journey extraction", "Grounded evidence + audit", "Formal score not yet reported"],
], columns=["method", "task", "evidence_required", "current_status"])

display(comparison)
checkpoint("Blank scores are not zero: unrun or inapplicable metrics stay unreported.")


The table is a **reporting schema**, not a leaderboard. Numerical columns should be added only after the held-out benchmark is frozen and the relevant method is run. This prevents worked examples from turning into accidental benchmark claims.

## 5. A lightweight adjudication exercise

Choose one disagreement and record your decision. The objective is not to maximize agreement with the machines; it is to make the scholarly decision explicit.

### Spoken adjudication exercise

For **Cambridge**, methods A and B return `GPE`; method C returns no span. Decide:

1. accept the majority annotation;
2. edit its label or boundary;
3. reject it; or
4. defer for contextual review.

Write down one sentence of evidence for your choice. We will compare reasons, not merely votes.

Questions to discuss:

1. Would your decision change if the text were eighteenth-century rather than contemporary?
2. Would your decision change if the next sentence mentioned Massachusetts?
3. Is majority vote a method of truth, or merely a routing heuristic?
4. Which disagreements deserve mandatory human review?

## 6. Export the comparison skeleton

This tidy structure is designed to feed the later benchmark runner, Streamlit comparison view and keynote figures.

In [ ]:
from pathlib import Path

out_dir = Path("sh2026_outputs/comparisons")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "comparison_template.csv"
comparison.to_csv(out_path, index=False)
print(out_path)

## 7. Take-away

A hybrid workflow is not simply "use many models". It needs explicit rules for:

**what is compared → how disagreement is represented → when a human intervenes → how that intervention is recorded → how burden is measured**

This makes human review part of the experimental design rather than an invisible clean-up step.

**Next:** move from annotations to maps and other spatial representations while preserving uncertainty and provenance.